# 02 - Models and evaluation (final, reproducible)
Run top to bottom (Restart and run all, ~2 minutes). Needs `processed/jobs_final.pkl`, `skill_vocab_final.pkl`, `p_it_skills_v1.pkl`
and the four validation files, plus `career_utils.py`, `career_pipeline.py`, `recommender.py` in the project folder.

Every number printed here goes into the report. Protocol: one **company-wise** 80/20 split (a company's listings never sit on both sides);
all model choices are made by 5-fold CV on the 80% part; the 20% is used once for the final numbers.

In [1]:
import os, sys, pickle, time
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path("/content/drive/MyDrive/Job recommendation nd skill gap analysis")
except ImportError:
    BASE = Path(os.environ["CAREER_BASE"])
sys.path.insert(0, str(BASE))
import numpy as np, pandas as pd
from collections import Counter
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity
from career_utils import OTHER, AMBIG
from recommender import CareerRecommender, GENERIC_SKILLS

PROC = BASE / "processed"; BASE_PROC = str(PROC) + "/"
ext = pd.read_pickle(PROC / "jobs_final.pkl")
skill_vocab = pickle.load(open(PROC / "skill_vocab_final.pkl", "rb"))
print(ext.shape, "| tiers:", ext["it_tier"].value_counts().to_dict())

Mounted at /content/drive
(93729, 31) | tiers: {'non_IT_or_unknown': 60119, 'core_rule': 21401, 'extended_mid': 6235, 'extended_high': 4461, 'core_ambiguous': 1513}


## 1. Family classifier
### 1.1 Company-wise split (deterministic folds)

In [2]:
def group_folds(groups, k=5):
    """Deterministic, version-independent GroupKFold: biggest companies first, each to the lightest fold."""
    sizes = groups.value_counts()
    order = sorted(sizes.items(), key=lambda kv: (-kv[1], str(kv[0])))
    load, assign = [0] * k, {}
    for g, n in order:
        i = int(np.argmin(load)); assign[g] = i; load[i] += n
    return groups.map(assign).to_numpy()

ev = ext[ext["family_source"].eq("rule")]
ev = ev[ev["skills_list"].map(len) >= 3].reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_i, te_i = next(gss.split(ev, groups=ev["companyName"].fillna("unk")))
tr, te = ev.iloc[tr_i].reset_index(drop=True), ev.iloc[te_i].reset_index(drop=True)
print("core rows:", len(ev), "| train:", len(tr), "| test:", len(te), "| families:", ev["family"].nunique())

core rows: 21256 | train: 17799 | test: 3457 | families: 15


### 1.2 Model choice by 5-fold company-wise CV on the training part
Same folds for every variant, so differences are paired ("better in x/5 folds").

In [3]:
def cv_folds(data, use_desc=False, class_weight=None, k=5):
    fold = group_folds(data["companyName"].fillna("unk"), k)
    t1, t3, f1 = [], [], []
    for f in range(k):
        a, b = data[fold != f], data[fold == f]
        cvz = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
        Xa, Xb = cvz.fit_transform(a["skills_list"]), cvz.transform(b["skills_list"])
        if use_desc:
            tf = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
            Xa = hstack([Xa, tf.fit_transform(a["description_clean"])]).tocsr()
            Xb = hstack([Xb, tf.transform(b["description_clean"])]).tocsr()
        m = LogisticRegression(max_iter=1000, class_weight=class_weight).fit(Xa, a["family"])
        P, cls = m.predict_proba(Xb), m.classes_
        o = np.argsort(-P, axis=1); y = b["family"].to_numpy()
        t1.append((cls[o[:, 0]] == y).mean()); t3.append(np.mean([y[i] in cls[o[i, :3]] for i in range(len(y))]))
        f1.append(f1_score(y, cls[o[:, 0]], average="macro"))
    return {"top1": np.array(t1), "top3": np.array(t3), "f1": np.array(f1)}
t=time.time()
res = {"1) skills, balanced weights": cv_folds(tr, class_weight="balanced"),
       "2) skills, no weights": cv_folds(tr),
       "3) skills + description, no weights": cv_folds(tr, use_desc=True)}
base = res["1) skills, balanced weights"]; rows = []
for name, v in res.items():
    d1, dF = v["top1"] - base["top1"], v["f1"] - base["f1"]
    rows.append({"variant": name, "top1": round(v["top1"].mean(), 3), "top3": round(v["top3"].mean(), 3), "macro_F1": round(v["f1"].mean(), 3),
                 "d_top1_vs_1": f"{d1.mean():+.3f} ({(d1 > 0).sum()}/5 folds)", "d_F1_vs_1": f"{dF.mean():+.3f} ({(dF > 0).sum()}/5 folds)"})
cv_table = pd.DataFrame(rows); print(cv_table.to_string(index=False)); print('secs', round(time.time()-t))
for n,v in res.items(): print(n, np.round(v['top1'],3).tolist())

                            variant  top1  top3  macro_F1        d_top1_vs_1          d_F1_vs_1
        1) skills, balanced weights 0.704 0.912     0.651 +0.000 (0/5 folds) +0.000 (0/5 folds)
              2) skills, no weights 0.754 0.935     0.674 +0.050 (5/5 folds) +0.023 (4/5 folds)
3) skills + description, no weights 0.803 0.955     0.722 +0.099 (5/5 folds) +0.071 (5/5 folds)
secs 121
1) skills, balanced weights [0.524, 0.778, 0.754, 0.725, 0.738]
2) skills, no weights [0.622, 0.803, 0.773, 0.773, 0.798]
3) skills + description, no weights [0.715, 0.846, 0.818, 0.803, 0.832]


### 1.3 Final models on the held-out companies (used once)
Model S = skills only (for resumes), Model D = skills + description (for job postings).

In [4]:
def fit_S_D(train):
    cv_s = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
    Xs = cv_s.fit_transform(train["skills_list"])
    model_s = LogisticRegression(max_iter=1000).fit(Xs, train["family"])
    cv_d = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3)
    tf_d = TfidfVectorizer(max_features=20000, min_df=5, sublinear_tf=True, ngram_range=(1, 2), stop_words="english")
    Xd = hstack([cv_d.fit_transform(train["skills_list"]), tf_d.fit_transform(train["description_clean"])]).tocsr()
    model_d = LogisticRegression(max_iter=1000).fit(Xd, train["family"])
    return cv_s, model_s, cv_d, tf_d, model_d

def score(model, X, y_true):
    P, cls = model.predict_proba(X), model.classes_
    o = np.argsort(-P, axis=1)
    return {"top1_acc": round((cls[o[:, 0]] == y_true).mean(), 3),
            "top3_acc": round(np.mean([y_true[i] in cls[o[i, :3]] for i in range(len(y_true))]), 3),
            "macro_F1": round(f1_score(y_true, cls[o[:, 0]], average="macro"), 3)}

cv_s, model_s, cv_d, tf_d, model_d = fit_S_D(tr)
y_te = te["family"].to_numpy()
Xte_d = hstack([cv_d.transform(te["skills_list"]), tf_d.transform(te["description_clean"])]).tocsr()
test_table = pd.DataFrame([
    {"model": "S: skills only", **score(model_s, cv_s.transform(te["skills_list"]), y_te)},
    {"model": "D: skills + description", **score(model_d, Xte_d, y_te)}])
print(test_table.to_string(index=False))
from sklearn.metrics import classification_report
print("\nPer-family (model D, held-out companies):")
print(classification_report(y_te, model_d.classes_[model_d.predict_proba(Xte_d).argmax(1)], digits=3))

                  model  top1_acc  top3_acc  macro_F1
         S: skills only     0.795     0.939     0.711
D: skills + description     0.823     0.960     0.749

Per-family (model D, held-out companies):
                            precision    recall  f1-score   support

Business / Systems Analyst      0.701     0.684     0.692        79
             Cybersecurity      0.906     0.795     0.847        73
         Data / BI Analyst      0.737     0.724     0.730        58
             Data Engineer      0.835     0.820     0.827       283
    Data Scientist / ML-AI      0.849     0.763     0.804       177
            Database Admin      0.800     0.711     0.753        45
            DevOps / Cloud      0.726     0.731     0.728       167
      ERP / CRM Consultant      0.921     0.930     0.926       669
            Frontend / Web      0.711     0.524     0.604       164
              IT Architect      0.528     0.339     0.413        56
IT Support / Sys & Network      0.824     0.70

## 2. Human validation (180 + 80 hand-labelled listings)
A: do the title rules agree with a human? B: is the model right on ambiguous titles? C: how many IT jobs hide in the 'Other' bucket (stratified estimate)?

In [5]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (float(round(c - h, 3)), float(round(c + h, 3)))

def norm_id(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.lstrip("0")

real_fams = sorted(f for f in ext["family"].unique() if f not in (OTHER, AMBIG))
allowed = set(real_fams) | {"IT - other/unclear", "Not IT"}
lab1 = pd.read_csv(BASE_PROC + "validation_labeled.csv")[["row_id", "your_family"]]
key1 = pd.read_csv(BASE_PROC + "validation_KEY_do_not_open.csv")
m1 = lab1.merge(key1, on="row_id"); m1["your_family"] = m1["your_family"].fillna("").astype(str).str.strip()
assert m1["your_family"].isin(allowed).all(), "unrecognised labels in sheet 1"

a = m1[m1["sheet"].eq("A_rule_label")]
k, n = int((a["your_family"] == a["label_to_check"]).sum()), len(a)
kn = int((a["your_family"] == "Not IT").sum())
A_res = {"agree": k, "n": n, "rate": round(k / n, 3), "ci": wilson(k, n), "not_IT": kn}
print(f"A) Rule-based family labels vs human: {k}/{n} = {k/n:.3f}  95% CI {wilson(k, n)} | labelled 'Not IT': {kn}/{n}")
b = m1[m1["sheet"].eq("B_ambiguous")]
kb, nb = int((b["your_family"] == b["label_to_check"]).sum()), len(b)
print(f"B) Model D on ambiguous postings vs human: {kb}/{nb} = {kb/nb:.3f}  95% CI {wilson(kb, nb)}")
sd_h = (b["your_family"] == "Software Developer").mean(); sd_m = (b["label_to_check"] == "Software Developer").mean()
print(f"   Software Developer share: model {sd_m:.2f} vs human {sd_h:.2f}")

# --- C + stratified estimate of IT listings hiding in 'Other'
edges = [0, 0.05, 0.15, 0.5, 0.9, 1.0001]; names = ["p<0.05", "0.05-0.15", "0.15-0.5", "0.5-0.9", ">=0.9"]
lab2 = pd.read_csv(BASE_PROC + "validation2_labeled.csv")[["row_id", "your_family"]]
key2 = pd.read_csv(BASE_PROC + "validation2_KEY_do_not_open.csv")
m2 = lab2.merge(key2, on="row_id"); m2["your_family"] = m2["your_family"].fillna("").astype(str).str.strip()
assert m2["your_family"].isin(allowed).all(), "unrecognised labels in sheet 2"
m2["is_it"] = m2["your_family"].isin(real_fams).astype(int)
m2["bin"] = pd.cut(m2["p_it_skills"], edges, labels=names, right=False)
samples = m2[["bin", "is_it"]]
c30 = m1[m1["sheet"].eq("C_other")].copy(); c30["jid"] = norm_id(c30["jobId"])
sc = ext[["jobId", "p_it_skills"]].assign(jid=lambda d: norm_id(d["jobId"]))
dev = c30.merge(sc[["jid", "p_it_skills"]], on="jid")
dev["is_it"] = dev["your_family"].isin(real_fams).astype(int)
dev["bin"] = pd.cut(dev["p_it_skills"], edges, labels=names, right=False)
samples = pd.concat([samples, dev[["bin", "is_it"]]])          # 30 earlier random rows are uniform samples too
print(f"C) first random sample of 'Other': {int(dev['is_it'].sum())}/{len(dev)} were real IT")

oth = ext[ext["family"].eq(OTHER)]
N_bin = pd.cut(oth["p_it_skills"], edges, labels=names, right=False).value_counts().reindex(names)
rows = []
for bn in names:
    s = samples[samples["bin"] == bn]; n_, k_ = len(s), int(s["is_it"].sum())
    rows.append({"bin": bn, "N_in_bin": int(N_bin[bn]), "n_labelled": n_, "IT_found": k_, "IT_rate": round(k_ / n_, 3),
                 "rate_95CI": wilson(k_, n_), "est_IT_jobs": int(round(k_ / n_ * N_bin[bn]))})
tab = pd.DataFrame(rows); print("\n", tab.to_string(index=False))
rng = np.random.default_rng(0); tot = np.zeros(20000)
for r in rows:
    tot += rng.beta(r["IT_found"] + 0.5, r["n_labelled"] - r["IT_found"] + 0.5, 20000) * r["N_in_bin"]
lo, hi = np.percentile(tot, [2.5, 97.5]); missed = sum(r["est_IT_jobs"] for r in rows)
n_rule_it = int(ext["family"].ne(OTHER).sum())
print(f"\nIT listings hiding in 'Other': ~{missed:,} (95% interval {int(lo):,} to {int(hi):,}) of {len(oth):,}")
print(f"=> title-rule recall ~ {n_rule_it / (n_rule_it + missed):.2f} (rule false positives ignored)")
cuts = [("p>=0.05", names[1:]), ("p>=0.15", names[2:]), ("p>=0.5", names[3:]), ("p>=0.9", names[4:])]
out = []
for nm, bl in cuts:
    fl = sum(r["N_in_bin"] for r in rows if r["bin"] in bl); hit = sum(r["est_IT_jobs"] for r in rows if r["bin"] in bl)
    out.append({"cutoff": nm, "flagged": fl, "est_true_IT": hit, "precision": round(hit / fl, 3), "recall_of_missed": round(hit / missed, 3)})
cut_table = pd.DataFrame(out); print("\n", cut_table.to_string(index=False))

A) Rule-based family labels vs human: 102/120 = 0.850  95% CI (0.775, 0.903) | labelled 'Not IT': 2/120
B) Model D on ambiguous postings vs human: 17/30 = 0.567  95% CI (0.392, 0.726)
   Software Developer share: model 0.67 vs human 0.27
C) first random sample of 'Other': 7/30 were real IT

       bin  N_in_bin  n_labelled  IT_found  IT_rate      rate_95CI  est_IT_jobs
   p<0.05     51560          38         2    0.053 (0.015, 0.173)         2714
0.05-0.15      8559          17         5    0.294 (0.133, 0.531)         2517
 0.15-0.5      6235          22        17    0.773 (0.566, 0.899)         4818
  0.5-0.9      3820          17        16    0.941   (0.73, 0.99)         3595
    >=0.9       641          16        16    1.000   (0.806, 1.0)          641

IT listings hiding in 'Other': ~14,285 (95% interval 11,165 to 20,098) of 70,815
=> title-rule recall ~ 0.62 (rule false positives ignored)

  cutoff  flagged  est_true_IT  precision  recall_of_missed
p>=0.05    19255        11571  

## 3. Gap recommender: do the suggested skills match what similar real listings ask for?
Hide 40% of a listing's skills, recommend 8, count how many hidden ones are recovered (3 seeds).

In [6]:
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from recommender import GENERIC_SKILLS

def gap_hit_eval(n_eval=1000, k=200, topn=8, seed=0):
    rng = np.random.RandomState(seed)
    cand = te[te["skills_list"].map(len) >= 5]
    cand = cand.sample(min(n_eval, len(cand)), random_state=seed)
    cache, pop = {}, {}
    for fam in cand["family"].unique():
        sub = tr[tr["family"].eq(fam)]
        cache[fam] = (sub, cv_s.transform(sub["skills_list"]))
        pop[fam] = Counter(s for sk in sub["skills_list"] for s in set(sk))
    glob = Counter(s for sk in tr["skills_list"] for s in set(sk))
    res = {"personalized (kNN)": [], "family popularity": [], "global popularity": []}
    for _, row in cand.iterrows():
        skills = [s for s in row["skills_list"] if s in cv_s.vocabulary_]
        if len(skills) < 5:
            continue
        perm = rng.permutation(len(skills)); cut = int(round(0.6 * len(skills)))
        shown = [skills[i] for i in perm[:cut]]
        hidden = {skills[i] for i in perm[cut:]} - GENERIC_SKILLS
        if not hidden:
            continue
        sub, Xsub = cache[row["family"]]
        sims = cosine_similarity(cv_s.transform([sorted(shown)]), Xsub)[0]
        nn = sub.iloc[np.argsort(-sims)[:k]]
        c = Counter(s for sk in nn["skills_list"] for s in set(sk))
        for name, counter in [("personalized (kNN)", c), ("family popularity", pop[row["family"]]), ("global popularity", glob)]:
            rec = [s for s, _ in counter.most_common(80)
                   if s not in shown and s not in GENERIC_SKILLS and s in cv_s.vocabulary_][:topn]
            hit = len(set(rec) & hidden)
            res[name].append((hit / len(hidden), int(hit > 0)))
    return {n: {"recall": np.mean([r for r, _ in v]), "hit": np.mean([h for _, h in v]), "n": len(v)} for n, v in res.items()}

rows = []
for seed in (0, 1, 2):
    r = gap_hit_eval(seed=seed)
    for name, v in r.items():
        rows.append({"seed": seed, "method": name, "recall_of_hidden": round(v["recall"], 3), "hit_rate": round(v["hit"], 3), "n": v["n"]})
gap_df = pd.DataFrame(rows)
print(gap_df.pivot(index="method", columns="seed", values="recall_of_hidden").assign(mean=lambda d: d.mean(axis=1).round(3)).to_string())
print()
print(gap_df.groupby("method")[["recall_of_hidden", "hit_rate"]].mean().round(3).to_string())

seed                    0      1      2   mean
method                                        
family popularity   0.227  0.246  0.240  0.238
global popularity   0.107  0.111  0.113  0.110
personalized (kNN)  0.338  0.363  0.350  0.350

                    recall_of_hidden  hit_rate
method                                        
family popularity              0.238     0.464
global popularity              0.110     0.259
personalized (kNN)             0.350     0.607


## 4. Sensitivity: core only vs core + predicted-family listings (alpha = 0.5)

In [7]:
from sklearn.model_selection import GroupShuffleSplit as _GSS
core_rows = ext[ext["family_source"].eq("rule")]
pred_rows = ext[ext["family_source"].eq("predicted")]
fams = sorted(core_rows["family"].unique())

def top_skills(frame, fam, k=15, min_count=10):
    sub = frame[frame["family_final"].eq(fam)]
    c = Counter(s for sk in sub["skills_list"] for s in set(sk))
    return [s for s, v in c.most_common(80) if v >= min_count and s not in GENERIC_SKILLS][:k], len(sub)

ext_all = pd.concat([core_rows, pred_rows]); rows = []
for fam in fams:
    a, na = top_skills(core_rows, fam); b, nb = top_skills(ext_all, fam)
    rows.append({"family": fam, "n_core": na, "n_extended": nb, "top15_overlap": len(set(a) & set(b))})
p1 = pd.DataFrame(rows); print("Part 1 - top-15 demanded skills per family, core vs core+predicted")
print(p1.to_string(index=False)); print("mean overlap:", round(p1["top15_overlap"].mean(), 1), "/ 15")

base_rows = core_rows[core_rows["skills_list"].map(len) >= 5]
tr_i2, te_i2 = next(_GSS(n_splits=1, test_size=0.2, random_state=42).split(base_rows, groups=base_rows["companyName"].fillna("unk")))
tr_core, te_core = base_rows.iloc[tr_i2], base_rows.iloc[te_i2]
test_companies = set(te_core["companyName"].fillna("unk"))
pred_pool = pred_rows[~pred_rows["companyName"].fillna("unk").isin(test_companies)]
cvz = CountVectorizer(analyzer=lambda x: x, binary=True, min_df=3).fit(tr_core["skills_list"])
vocab2 = cvz.vocabulary_

def run_pool(pool_frames, n_eval=1000, k=200, topn=8, seed=0):
    pool = pd.concat(pool_frames); rng = np.random.RandomState(seed)
    cand = te_core.sample(min(n_eval, len(te_core)), random_state=seed); cache = {}
    for fam in cand["family"].unique():
        sub = pool[pool["family_final"].eq(fam)]; cache[fam] = (sub, cvz.transform(sub["skills_list"]))
    hits, anyhit = [], []
    for _, row in cand.iterrows():
        sk = [s for s in row["skills_list"] if s in vocab2]
        if len(sk) < 5: continue
        perm = rng.permutation(len(sk)); cut = int(round(0.6 * len(sk)))
        shown = [sk[i] for i in perm[:cut]]; hidden = {sk[i] for i in perm[cut:]} - GENERIC_SKILLS
        if not hidden: continue
        sub, X = cache[row["family"]]
        sims = cosine_similarity(cvz.transform([sorted(shown)]), X)[0]
        nn = sub.iloc[np.argsort(-sims)[:k]]
        c = Counter(s for l in nn["skills_list"] for s in set(l))
        rec = [s for s, _ in c.most_common(80) if s not in shown and s not in GENERIC_SKILLS and s in vocab2][:topn]
        h = len(set(rec) & hidden); hits.append(h / len(hidden)); anyhit.append(int(h > 0))
    return round(np.mean(hits), 3), round(np.mean(anyhit), 3)

tr_core = tr_core.assign(family_final=tr_core["family"]); res = []
for seed in (0, 1, 2):
    a = run_pool([tr_core], seed=seed); b = run_pool([tr_core, pred_pool], seed=seed)
    res.append({"seed": seed, "core_only_recall": a[0], "core_only_hit": a[1], "core+predicted_recall": b[0], "core+predicted_hit": b[1]})
p2 = pd.DataFrame(res); print("\nPart 2 - gap recommender, neighbour pool = core only vs core + predicted listings"); print(p2.to_string(index=False))

Part 1 - top-15 demanded skills per family, core vs core+predicted
                    family  n_core  n_extended  top15_overlap
Business / Systems Analyst     514        1035             13
             Cybersecurity     500         929             12
         Data / BI Analyst     485        1131             12
             Data Engineer    1444        1888             15
    Data Scientist / ML-AI    1093        1592             15
            Database Admin     231         390             13
            DevOps / Cloud    1047        1715             14
      ERP / CRM Consultant    4013        5534             12
            Frontend / Web     802        1170             14
              IT Architect     515         894              7
IT Support / Sys & Network     955        1896             12
          Mobile Developer     257         357             14
              QA / Testing    1256        2313             13
        Software Developer    7909       10295             13
   

## 5. The recommender in action (personas)
Trained on the core listings; the gap and roadmap use the core + predicted listings.

In [8]:
core_df = ext[ext["family_source"].eq("rule")]
market_df = ext[ext["family_source"].isin(["rule", "predicted"])]
rec = CareerRecommender(core_df, market_df, skill_vocab)
PERSONAS = {
    "Backend fresher": ["java", "spring boot", "sql", "mysql", "git", "rest apis", "microservices"],
    "Tester":          ["selenium", "manual testing", "automation testing", "java", "sql", "jira"],
    "Data engineer":   ["python", "sql", "pyspark", "airflow", "etl", "hive", "amazon web services"],
    "Security":        ["cyber security", "network security", "siem", "owasp", "linux"],
    "Web frontend":    ["javascript", "react.js", "html", "css", "typescript"],
}
for name, sk in PERSONAS.items():
    print(f"\n==================== {name}"); rec.print_report(sk)
print("\nSkills read from free text:", rec.skills_from_text("2 years of Python, pandas and SQL, some AWS and Docker; built ML models with scikit-learn."))


==================== Backend fresher
Skills recognised: 7 of 7

#1 Software Developer | prob 0.95 | lift x2.6
   learn next: angular (32.0%), react.js (31.5%), hibernate (18.5%), full stack (15.0%), amazon web services (9.5%), bootstrap (8.5%), kafka (7.0%), css (5.0%)
   roadmap: javascript -> hibernate -> html -> angular -> full stack -> linux -> bootstrap -> kafka -> css -> amazon web services -> react.js

==================== Tester
Skills recognised: 6 of 6

#1 QA / Testing | prob 0.85 | lift x14.5
   learn next: api testing (22.5%), regression testing (16.0%), continuous integration/continuous delivery (12.5%), cucumber (11.5%), appium (11.5%), python (8.5%), test cases (8.5%), quality assurance (7.5%)
   roadmap: python -> quality assurance -> test cases -> api testing -> regression testing -> continuous integration/continuous delivery -> appium -> cucumber

#2 Software Developer | prob 0.10 | lift x0.3
   learn next: spring boot (37.5%), microservices (32.5%), angular (20.0%),

## 6. Why O*NET's RIASEC interests are not used to rank IT careers
The RIASEC vectors of IT occupations are nearly identical, so they cannot separate the families.

In [9]:
onet_dir = BASE / "onet"
if (onet_dir / "career_interest_types.csv").exists():
    ci = pd.read_csv(onet_dir / "career_interest_types.csv"); oi = ci[ci["Scale ID"] == "OI"]
    letters = ["Realistic", "Investigative", "Artistic", "Social", "Enterprising", "Conventional"]
    piv = oi.pivot_table(index="O*NET-SOC Code", columns="Element Name", values="Data Value")[letters]
    codes = {"Software Developer": "15-1252.00", "Frontend / Web": "15-1254.00", "Data Engineer": "15-1243.00",
             "Data Scientist": "15-2051.00", "Data / BI": "15-2051.01", "Database Admin": "15-1242.00",
             "DevOps / Cloud": "15-1244.00", "QA / Testing": "15-1253.00", "IT Support": "15-1232.00",
             "Cybersecurity": "15-1212.00", "ERP / IT Architect": "15-1299.08", "Business Analyst": "15-1211.00",
             "Tech Project": "15-1299.09"}
    M_ = piv.loc[list(codes.values())].to_numpy(); names_ = list(codes)
    Nn = M_ / np.linalg.norm(M_, axis=1, keepdims=True); Cs = Nn @ Nn.T
    iu = np.triu_indices(len(names_), 1)
    riasec_res = {"mean": round(float(Cs[iu].mean()), 3), "min": round(float(Cs[iu].min()), 3), "max": round(float(Cs[iu].max()), 3)}
    print("Pairwise cosine between the RIASEC vectors of the IT occupations:", riasec_res)
    sw = piv.loc["15-1252.00"].to_numpy()
    for label, code in [("Registered nurse", "29-1141.00"), ("Teacher", "25-2031.00"), ("Sales rep", "41-3091.00")]:
        if code in piv.index:
            v = piv.loc[code].to_numpy(); print(f"  Software Developer vs {label}: {float(sw @ v / (np.linalg.norm(sw) * np.linalg.norm(v))):.3f}")
else:
    print("O*NET files not found - skipped")

Pairwise cosine between the RIASEC vectors of the IT occupations: {'mean': 0.956, 'min': 0.843, 'max': 0.997}
  Software Developer vs Registered nurse: 0.905
  Software Developer vs Teacher: 0.782
  Software Developer vs Sales rep: 0.721


## 7. Save all result tables (they feed the report)

In [10]:
RES = PROC / "results"; RES.mkdir(exist_ok=True)
cv_table.to_csv(RES / "family_cv_5fold.csv", index=False)
test_table.to_csv(RES / "family_heldout_test.csv", index=False)
tab.to_csv(RES / "other_bucket_stratified_estimate.csv", index=False)
cut_table.to_csv(RES / "detector_cutoffs.csv", index=False)
gap_df.to_csv(RES / "gap_recommender_eval.csv", index=False)
p1.to_csv(RES / "sensitivity_top_skills.csv", index=False)
p2.to_csv(RES / "sensitivity_gap_recommender.csv", index=False)
import json
json.dump({"A_rule_vs_human": A_res, "B_modelD_ambiguous": {"agree": kb, "n": nb, "ci": wilson(kb, nb)},
           "missed_IT_in_Other": {"estimate": int(missed), "ci": [int(lo), int(hi)]},
           "riasec_cosine": riasec_res if "riasec_res" in globals() else None},
          open(RES / "human_validation_summary.json", "w"), indent=2)
print("saved to", RES, ":", sorted(p.name for p in RES.iterdir()))

saved to /content/drive/MyDrive/Job recommendation nd skill gap analysis/processed/results : ['detector_cutoffs.csv', 'family_cv_5fold.csv', 'family_heldout_test.csv', 'gap_recommender_eval.csv', 'human_validation_summary.json', 'other_bucket_stratified_estimate.csv', 'sensitivity_gap_recommender.csv', 'sensitivity_top_skills.csv']
